# 모델 학습 코드 (xgboost, lightgbm, catboost, random forest, logistic)

## XGBoost

In [1]:
import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score
from xgboost import XGBClassifier

# ==========================================
# 1. 데이터 불러오기 (트리 기반 모델용 unscaled)
# ==========================================
# XGBoost는 트리 기반 모델이므로 스케일링이 안 된 데이터를 부릅니다.
print("📥 데이터를 불러오고 K-Fold를 위해 병합합니다...")

# 기존에 쪼개져 있던 Train과 Val 데이터를 불러옵니다.
X_train_part = pd.read_csv('X_tr_unscaled.csv')
X_val_part = pd.read_csv('X_val_unscaled.csv')

y_train_part = pd.read_csv('y_tr.csv')['returned']
y_val_part = pd.read_csv('y_val.csv')['returned']

# K-Fold가 전체 데이터에서 자유롭게 쪼갤 수 있도록 하나로 합칩니다.
X = pd.concat([X_train_part, X_val_part], axis=0).reset_index(drop=True)
y = pd.concat([y_train_part, y_val_part], axis=0).reset_index(drop=True)

X_test = pd.read_csv('X_test_unscaled.csv')
test_id_df = pd.read_csv('test_order_id.csv')

print(f"✅ 병합 완료! (전체 학습 데이터 X: {X.shape}, y: {y.shape})")

# ==========================================
# 🌟 1.5 K-Fold 시작 전 Optuna 하이퍼파라미터 탐색 추가
# ==========================================
import optuna
from sklearn.model_selection import train_test_split

print("\n🔍 Optuna 탐색을 시작합니다 (탐색 속도를 위해 임시 8:2 분할)...")
# 전체 X, y를 Optuna가 빨리 학습해볼 수 있게 임시로 분할합니다.
X_opt_tr, X_opt_val, y_opt_tr, y_opt_val = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
scale_weight_opt = (y_opt_tr == 0).sum() / (y_opt_tr == 1).sum()

def objective(trial):
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 300, 1000),
        'max_depth': trial.suggest_int('max_depth', 4, 10),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.1, log=True),
        'subsample': trial.suggest_float('subsample', 0.6, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 1.0),
        'gamma': trial.suggest_float('gamma', 0.0, 1.0),
        'reg_lambda': trial.suggest_float('reg_lambda', 0.0, 5.0),
        'scale_pos_weight': scale_weight_opt,
        'random_state': 42,
        'n_jobs': -1,
        'tree_method': 'hist',
        'eval_metric': 'logloss'
    }
    model_opt = XGBClassifier(**params)
    model_opt.fit(X_opt_tr, y_opt_tr, eval_set=[(X_opt_val, y_opt_val)], verbose=False)
    return roc_auc_score(y_opt_val, model_opt.predict_proba(X_opt_val)[:, 1])

# 30회 탐색 실행
study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=30)
print(f"🎉 탐색 완료! 최고 Validation AUC: {study.best_value:.4f}")

# ==========================================
# 2. K-Fold 설정 및 XGBoost 모델 학습
# ==========================================
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

oof_pred = np.zeros(len(X))          # 5번 검증한 결과를 모아둘 빈 배열
test_pred = np.zeros(len(X_test))    # 5번 예측한 테스트 결과를 누적할 빈 배열
fold_scores = []

print("\n🚀 XGBoost K-Fold 학습을 시작합니다...")

for fold, (train_idx, val_idx) in enumerate(skf.split(X, y), 1):
    # Fold별 데이터 분할
    X_tr, X_va = X.iloc[train_idx], X.iloc[val_idx]
    y_tr, y_va = y.iloc[train_idx], y.iloc[val_idx]
    
    # 기존에 세팅하셨던 모델 파라미터 그대로 적용
    # Fold별 클래스 불균형 비율 계산
    scale_pos_weight_fold = (y_tr == 0).sum() / (y_tr == 1).sum()

    # 🌟 Optuna가 위에서 찾아낸 최적의 조합을 덮어씌웁니다!
    model = XGBClassifier(
        **study.best_params,                    # 옵튜나 최적 파라미터 한방에 주입
        scale_pos_weight=scale_pos_weight_fold, # Fold별 불균형 비율 동적 적용
        random_state=42 + fold,                 # Fold마다 미세하게 다른 난수 부여
        n_jobs=-1,
        tree_method='hist',
        eval_metric='logloss'
    )
    
    # 조기 종료(Early Stopping)를 적용하여 과적합 방지
    model.fit(
        X_tr, y_tr,
        eval_set=[(X_va, y_va)],
        verbose=False
    )
    
    # 검증 데이터 예측 및 OOF 저장
    val_proba = model.predict_proba(X_va)[:, 1]
    oof_pred[val_idx] = val_proba
    
    # 테스트 데이터 예측 (5로 나누어 누적 합산 -> 최종적으로 평균이 됨)
    test_pred += model.predict_proba(X_test)[:, 1] / skf.n_splits
    
    # Fold별 점수 기록
    fold_auc = roc_auc_score(y_va, val_proba)
    fold_scores.append(fold_auc)
    print(f"   - Fold {fold} 완료 (ROC-AUC: {fold_auc:.4f})")

# ==========================================
# 3. 전체 OOF 성능 평가
# ==========================================
overall_auc = roc_auc_score(y, oof_pred)
# 기본 임계값 0.5를 기준으로 F1 및 정확도 계산
oof_label = (oof_pred >= 0.5).astype(int)
overall_f1 = f1_score(y, oof_label)
overall_acc = accuracy_score(y, oof_label)

print("\n--- 📊 최종 Validation(OOF) 결과 ---")
print(f"정확도 (Accuracy): {overall_acc:.4f}")
print(f"F1 점수 (F1 Score): {overall_f1:.4f}")
print(f"AUC (ROC-AUC):  {overall_auc:.4f}")
print(f"Fold별 AUC: {[round(s, 4) for s in fold_scores]}")


📥 데이터를 불러오고 K-Fold를 위해 병합합니다...
✅ 병합 완료! (전체 학습 데이터 X: (200000, 44), y: (200000,))


c:\Users\goiji\.conda\envs\goji60000-env\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
[I 2026-04-09 09:47:59,724] A new study created in memory with name: no-name-723995d6-fe28-49c1-9dab-797ddbb88e99



🔍 Optuna 탐색을 시작합니다 (탐색 속도를 위해 임시 8:2 분할)...


[I 2026-04-09 09:48:09,509] Trial 0 finished with value: 0.5943871003767162 and parameters: {'n_estimators': 789, 'max_depth': 7, 'learning_rate': 0.011285720261763076, 'subsample': 0.6040112011382368, 'colsample_bytree': 0.852327081247167, 'gamma': 0.6621610196013624, 'reg_lambda': 2.703045305593166}. Best is trial 0 with value: 0.5943871003767162.
[I 2026-04-09 09:48:20,151] Trial 1 finished with value: 0.5900482145243301 and parameters: {'n_estimators': 401, 'max_depth': 10, 'learning_rate': 0.01564124306968689, 'subsample': 0.6641512066144677, 'colsample_bytree': 0.6593774531609943, 'gamma': 0.7178875078388978, 'reg_lambda': 2.604398037914129}. Best is trial 0 with value: 0.5943871003767162.
[I 2026-04-09 09:48:34,403] Trial 2 finished with value: 0.5860350705444498 and parameters: {'n_estimators': 749, 'max_depth': 9, 'learning_rate': 0.01875663389217498, 'subsample': 0.699591811871892, 'colsample_bytree': 0.9744804844307904, 'gamma': 0.32807848081880897, 'reg_lambda': 1.700599249

🎉 탐색 완료! 최고 Validation AUC: 0.5971

🚀 XGBoost K-Fold 학습을 시작합니다...
   - Fold 1 완료 (ROC-AUC: 0.5962)
   - Fold 2 완료 (ROC-AUC: 0.5952)
   - Fold 3 완료 (ROC-AUC: 0.5989)
   - Fold 4 완료 (ROC-AUC: 0.5880)
   - Fold 5 완료 (ROC-AUC: 0.5938)

--- 📊 최종 Validation(OOF) 결과 ---
정확도 (Accuracy): 0.5663
F1 점수 (F1 Score): 0.5499
AUC (ROC-AUC):  0.5944
Fold별 AUC: [0.5962, 0.5952, 0.5989, 0.588, 0.5938]


In [3]:
import pandas as pd

print("🚀 대시보드용 CSV 파일 추출을 시작합니다...")

# ==========================================
# 1. Validation 예측 결과 (val_predictions.csv)
# ==========================================
# K-Fold의 OOF 결과(oof_pred)를 기존 val_proba 자리에 넣습니다.
val_df = pd.DataFrame({
    'pred_prob': oof_pred,
    'pred_label': oof_label,
    'returned': y.values 
})

val_df.to_csv('val_predictions.csv', index=False)
print("✅ [1/3] 'val_predictions.csv' 생성 완료!")

# ==========================================
# 2. Test 예측 결과 (test_predictions.csv)
# ==========================================
# 5번 누적 평균낸 test_pred를 확률로 쓰고, 0.5 기준으로 라벨을 만듭니다.
test_label = (test_pred >= 0.5).astype(int)

test_df = pd.DataFrame({
    'order_id': test_id_df['order_id'],
    'pred_prob': test_pred,
    'pred_label': test_label
})

test_df.to_csv('test_predictions.csv', index=False)
print("✅ [2/3] 'test_predictions.csv' 생성 완료!")

# ==========================================
# 3. 변수 중요도 (feature_importance.csv)
# ==========================================
importance_df = pd.DataFrame({
    'feature': X.columns,
    'importance': model.feature_importances_
})

# 중요도가 높은 순서대로 정렬
importance_df = importance_df.sort_values(by='importance', ascending=False)

importance_df.to_csv('feature_importance.csv', index=False)
print("✅ [3/3] 'feature_importance.csv' 생성 완료!")

# ---------------------------------------------------------
# 💡 (백그라운드 실행) 대시보드 에러 방지용 데이터 덮어쓰기
# K-Fold 결과와 길이가 맞지 않아 app3.py가 튕기는 것을 막아줍니다.
X.to_csv('X_val_unscaled.csv', index=False)
pd.DataFrame({'returned': y}).to_csv('y_val.csv', index=False)
# 캐글 제출용 파일도 하나 조용히 뽑아둡니다.
test_df[['order_id', 'pred_prob']].rename(columns={'pred_prob': 'returned'}).to_csv('xgb_submission.csv', index=False)
# ---------------------------------------------------------

print("\n🎉 모든 데이터 추출이 완료되었습니다! 대시보드 담당자에게 전달해 주세요.")

🚀 대시보드용 CSV 파일 추출을 시작합니다...
✅ [1/3] 'val_predictions.csv' 생성 완료!
✅ [2/3] 'test_predictions.csv' 생성 완료!
✅ [3/3] 'feature_importance.csv' 생성 완료!

🎉 모든 데이터 추출이 완료되었습니다! 대시보드 담당자에게 전달해 주세요.


In [2]:
# ==========================================
# 4. 대시보드(app3.py) 연동용 CSV 일괄 추출
# ==========================================
print("\n💾 대시보드 및 제출용 CSV 파일 추출을 시작합니다...")

# 1. Validation 예측 결과 (val_predictions.csv)
val_df = pd.DataFrame({
    'pred_prob': oof_pred,
    'pred_label': oof_label,
    'returned': y.values
})
val_df.to_csv('val_predictions.csv', index=False)

# 2. Test 예측 결과 (test_predictions.csv 및 제출용)
test_label = (test_pred >= 0.5).astype(int)
test_df = pd.DataFrame({
    'order_id': test_id_df['order_id'],
    'pred_prob': test_pred,
    'pred_label': test_label
})
test_df.to_csv('test_predictions.csv', index=False)
test_df[['order_id', 'pred_prob']].rename(columns={'pred_prob': 'returned'}).to_csv('xgb_submission.csv', index=False)

# 3. 변수 중요도 (feature_importance.csv) - 마지막 Fold 모델 기준
importance_df = pd.DataFrame({
    'feature': X.columns,
    'importance': model.feature_importances_
})
importance_df = importance_df.sort_values(by='importance', ascending=False)
importance_df.to_csv('feature_importance.csv', index=False)

# 4. 대시보드 에러 방지용 전체 원본 데이터 동기화
X.to_csv('X_val_unscaled.csv', index=False)
pd.DataFrame({'returned': y}).to_csv('y_val.csv', index=False)



💾 대시보드 및 제출용 CSV 파일 추출을 시작합니다...


## Lightgbm

In [3]:
import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score
from lightgbm import LGBMClassifier

# ==========================================
# 1. 데이터 불러오기 (트리 기반 모델용 unscaled)
# ==========================================
# XGBoost는 트리 기반 모델이므로 스케일링이 안 된 데이터를 부릅니다.
print("📥 데이터를 불러오고 K-Fold를 위해 병합합니다...")

# 기존에 쪼개져 있던 Train과 Val 데이터를 불러옵니다.
X_train_part = pd.read_csv('X_tr_unscaled.csv')
X_val_part = pd.read_csv('X_val_unscaled.csv')

y_train_part = pd.read_csv('y_tr.csv')['returned']
y_val_part = pd.read_csv('y_val.csv')['returned']

# K-Fold가 전체 데이터에서 자유롭게 쪼갤 수 있도록 하나로 합칩니다.
X = pd.concat([X_train_part, X_val_part], axis=0).reset_index(drop=True)
y = pd.concat([y_train_part, y_val_part], axis=0).reset_index(drop=True)

X_test = pd.read_csv('X_test_unscaled.csv')
test_id_df = pd.read_csv('test_order_id.csv')

print(f"✅ 병합 완료! (전체 학습 데이터 X: {X.shape}, y: {y.shape})")

# ==========================================
# 🌟 1.5 K-Fold 시작 전 Optuna 하이퍼파라미터 탐색 추가
# ==========================================
import optuna
from sklearn.model_selection import train_test_split

print("\n🔍 Optuna 탐색을 시작합니다 (탐색 속도를 위해 임시 8:2 분할)...")
# 전체 X, y를 Optuna가 빨리 학습해볼 수 있게 임시로 분할합니다.
X_opt_tr, X_opt_val, y_opt_tr, y_opt_val = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
scale_weight_opt = (y_opt_tr == 0).sum() / (y_opt_tr == 1).sum()

def objective(trial):
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 300, 1000),
        'max_depth': trial.suggest_int('max_depth', 4, 10),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.1, log=True),
        'num_leaves': trial.suggest_int('num_leaves', 20, 150),
        'subsample': trial.suggest_float('subsample', 0.6, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 1.0),
        'class_weight': 'balanced',
        'random_state': 42,
        'n_jobs': -1,
        'verbose': -1
    }
    
    # LightGBM 전용 과적합 방지 로직
    if params['num_leaves'] > (2 ** params['max_depth']):
        params['num_leaves'] = (2 ** params['max_depth']) - 1

    model_opt = LGBMClassifier(**params)
    model_opt.fit(X_opt_tr, y_opt_tr, eval_set=[(X_opt_val, y_opt_val)])
    return roc_auc_score(y_opt_val, model_opt.predict_proba(X_opt_val)[:, 1])

# 30회 탐색 실행
study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=30)
print(f"🎉 탐색 완료! 최고 Validation AUC: {study.best_value:.4f}")

# ==========================================
# 2. K-Fold 설정 및 lightGBM 모델 학습
# ==========================================
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

oof_pred = np.zeros(len(X))          # 5번 검증한 결과를 모아둘 빈 배열
test_pred = np.zeros(len(X_test))    # 5번 예측한 테스트 결과를 누적할 빈 배열
fold_scores = []

print("\n🚀 lightGMB K-Fold 학습을 시작합니다...")

for fold, (train_idx, val_idx) in enumerate(skf.split(X, y), 1):
    # Fold별 데이터 분할
    X_tr, X_va = X.iloc[train_idx], X.iloc[val_idx]
    y_tr, y_va = y.iloc[train_idx], y.iloc[val_idx]
    
    # 기존에 세팅하셨던 모델 파라미터 그대로 적용
    # Fold별 클래스 불균형 비율 계산
    scale_pos_weight_fold = (y_tr == 0).sum() / (y_tr == 1).sum()

    # 🌟 Optuna가 위에서 찾아낸 최적의 조합을 덮어씌웁니다!
    model = LGBMClassifier(
        **study.best_params,        # 옵튜나 최적 파라미터 한방에 주입
        class_weight='balanced',    # LightGBM용 불균형 처리 설정
        random_state=42 + fold,     # Fold마다 미세하게 다른 난수 부여
        n_jobs=-1,
        verbose=-1
    )    
    # 조기 종료(Early Stopping)를 적용하여 과적합 방지
    model.fit(
        X_tr, y_tr,
        eval_set=[(X_va, y_va)],
    )
    
    # 검증 데이터 예측 및 OOF 저장
    val_proba = model.predict_proba(X_va)[:, 1]
    oof_pred[val_idx] = val_proba
    
    # 테스트 데이터 예측 (5로 나누어 누적 합산 -> 최종적으로 평균이 됨)
    test_pred += model.predict_proba(X_test)[:, 1] / skf.n_splits
    
    # Fold별 점수 기록
    fold_auc = roc_auc_score(y_va, val_proba)
    fold_scores.append(fold_auc)
    print(f"   - Fold {fold} 완료 (ROC-AUC: {fold_auc:.4f})")

# ==========================================
# 3. 전체 OOF 성능 평가
# ==========================================
overall_auc = roc_auc_score(y, oof_pred)
# 기본 임계값 0.5를 기준으로 F1 및 정확도 계산
oof_label = (oof_pred >= 0.5).astype(int)
overall_f1 = f1_score(y, oof_label)
overall_acc = accuracy_score(y, oof_label)

print("\n--- 📊 최종 Validation(OOF) 결과 ---")
print(f"정확도 (Accuracy): {overall_acc:.4f}")
print(f"F1 점수 (F1 Score): {overall_f1:.4f}")
print(f"AUC (ROC-AUC):  {overall_auc:.4f}")
print(f"Fold별 AUC: {[round(s, 4) for s in fold_scores]}")


📥 데이터를 불러오고 K-Fold를 위해 병합합니다...


[I 2026-04-09 10:24:30,259] A new study created in memory with name: no-name-43392dbc-c3d6-48c9-a8a6-fb8f8470c81b


✅ 병합 완료! (전체 학습 데이터 X: (200000, 44), y: (200000,))

🔍 Optuna 탐색을 시작합니다 (탐색 속도를 위해 임시 8:2 분할)...


[I 2026-04-09 10:24:34,809] Trial 0 finished with value: 0.5901717608529276 and parameters: {'n_estimators': 918, 'max_depth': 9, 'learning_rate': 0.037021572614972194, 'num_leaves': 31, 'subsample': 0.640864418780511, 'colsample_bytree': 0.9549643492793369}. Best is trial 0 with value: 0.5901717608529276.
[I 2026-04-09 10:24:39,539] Trial 1 finished with value: 0.5836386211713396 and parameters: {'n_estimators': 529, 'max_depth': 8, 'learning_rate': 0.05553801336792844, 'num_leaves': 75, 'subsample': 0.996396563504427, 'colsample_bytree': 0.9388907436216827}. Best is trial 0 with value: 0.5901717608529276.
[I 2026-04-09 10:24:45,513] Trial 2 finished with value: 0.5936218392632764 and parameters: {'n_estimators': 739, 'max_depth': 6, 'learning_rate': 0.015308149801925656, 'num_leaves': 44, 'subsample': 0.6152706965354916, 'colsample_bytree': 0.9499950627509572}. Best is trial 2 with value: 0.5936218392632764.
[I 2026-04-09 10:24:50,585] Trial 3 finished with value: 0.593264514637053 a

🎉 탐색 완료! 최고 Validation AUC: 0.5966

🚀 lightGMB K-Fold 학습을 시작합니다...
   - Fold 1 완료 (ROC-AUC: 0.5960)
   - Fold 2 완료 (ROC-AUC: 0.5941)
   - Fold 3 완료 (ROC-AUC: 0.5982)
   - Fold 4 완료 (ROC-AUC: 0.5866)
   - Fold 5 완료 (ROC-AUC: 0.5932)

--- 📊 최종 Validation(OOF) 결과 ---
정확도 (Accuracy): 0.5656
F1 점수 (F1 Score): 0.5491
AUC (ROC-AUC):  0.5936
Fold별 AUC: [0.596, 0.5941, 0.5982, 0.5866, 0.5932]


## Catboost

In [7]:
import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score
from catboost import CatBoostClassifier

# ==========================================
# 1. 데이터 불러오기 (트리 기반 모델용 unscaled)
# ==========================================
# XGBoost는 트리 기반 모델이므로 스케일링이 안 된 데이터를 부릅니다.
print("📥 데이터를 불러오고 K-Fold를 위해 병합합니다...")

# 기존에 쪼개져 있던 Train과 Val 데이터를 불러옵니다.
X_train_part = pd.read_csv('X_tr_unscaled.csv')
X_val_part = pd.read_csv('X_val_unscaled.csv')

y_train_part = pd.read_csv('y_tr.csv')['returned']
y_val_part = pd.read_csv('y_val.csv')['returned']

# K-Fold가 전체 데이터에서 자유롭게 쪼갤 수 있도록 하나로 합칩니다.
X = pd.concat([X_train_part, X_val_part], axis=0).reset_index(drop=True)
y = pd.concat([y_train_part, y_val_part], axis=0).reset_index(drop=True)

X_test = pd.read_csv('X_test_unscaled.csv')
test_id_df = pd.read_csv('test_order_id.csv')

print(f"✅ 병합 완료! (전체 학습 데이터 X: {X.shape}, y: {y.shape})")

# ==========================================
# 🌟 1.5 K-Fold 시작 전 Optuna 하이퍼파라미터 탐색 추가
# ==========================================
import optuna
from sklearn.model_selection import train_test_split

print("\n🔍 Optuna 탐색을 시작합니다 (탐색 속도를 위해 임시 8:2 분할)...")
# 전체 X, y를 Optuna가 빨리 학습해볼 수 있게 임시로 분할합니다.
X_opt_tr, X_opt_val, y_opt_tr, y_opt_val = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
scale_weight_opt = (y_opt_tr == 0).sum() / (y_opt_tr == 1).sum()

def objective(trial):
    params = {
        'iterations': trial.suggest_int('iterations', 300, 1000),
        'depth': trial.suggest_int('depth', 4, 10),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.1, log=True),
        'l2_leaf_reg': trial.suggest_float('l2_leaf_reg', 1.0, 10.0),
        'auto_class_weights': 'Balanced',
        'random_state': 42,
        'verbose': 0  # 🌟 추가: CatBoost는 0으로 해야 로그가 숨겨집니다.
    }
    
    model_opt = CatBoostClassifier(**params)
    
    # verbose 관련 에러 방지를 위해 여기서는 eval_set만 남깁니다.
    model_opt.fit(X_opt_tr, y_opt_tr, eval_set=[(X_opt_val, y_opt_val)])
    
    return roc_auc_score(y_opt_val, model_opt.predict_proba(X_opt_val)[:, 1])

# 30회 탐색 실행
study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=30)
print(f"🎉 탐색 완료! 최고 Validation AUC: {study.best_value:.4f}")

# ==========================================
# 2. K-Fold 설정 및 XGBoost 모델 학습
# ==========================================
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

oof_pred = np.zeros(len(X))          # 5번 검증한 결과를 모아둘 빈 배열
test_pred = np.zeros(len(X_test))    # 5번 예측한 테스트 결과를 누적할 빈 배열
fold_scores = []

print("\n🚀 CatBoost K-Fold 학습을 시작합니다...")

for fold, (train_idx, val_idx) in enumerate(skf.split(X, y), 1):
    # Fold별 데이터 분할
    X_tr, X_va = X.iloc[train_idx], X.iloc[val_idx]
    y_tr, y_va = y.iloc[train_idx], y.iloc[val_idx]
    
    # 기존에 세팅하셨던 모델 파라미터 그대로 적용
    # Fold별 클래스 불균형 비율 계산
    scale_pos_weight_fold = (y_tr == 0).sum() / (y_tr == 1).sum()

    # 🌟 Optuna 결과 덮어쓰기
    model = CatBoostClassifier(
        **study.best_params,
        random_state=42 + fold,     # Fold마다 미세하게 다른 난수 부여
        verbose=0                   # 🌟 추가: 로그 숨김
    )
    
    # 🌟 fit 수정: verbose 관련 내용 제거
    model.fit(X_tr, y_tr, eval_set=[(X_va, y_va)])
    
    # 검증 데이터 예측 및 OOF 저장
    val_proba = model.predict_proba(X_va)[:, 1]
    oof_pred[val_idx] = val_proba
    
    # 테스트 데이터 예측 (5로 나누어 누적 합산 -> 최종적으로 평균이 됨)
    test_pred += model.predict_proba(X_test)[:, 1] / skf.n_splits
    
    # Fold별 점수 기록
    fold_auc = roc_auc_score(y_va, val_proba)
    fold_scores.append(fold_auc)
    print(f"   - Fold {fold} 완료 (ROC-AUC: {fold_auc:.4f})")

# ==========================================
# 3. 전체 OOF 성능 평가
# ==========================================
overall_auc = roc_auc_score(y, oof_pred)
# 기본 임계값 0.5를 기준으로 F1 및 정확도 계산
oof_label = (oof_pred >= 0.5).astype(int)
overall_f1 = f1_score(y, oof_label)
overall_acc = accuracy_score(y, oof_label)

print("\n--- 📊 최종 Validation(OOF) 결과 ---")
print(f"정확도 (Accuracy): {overall_acc:.4f}")
print(f"F1 점수 (F1 Score): {overall_f1:.4f}")
print(f"AUC (ROC-AUC):  {overall_auc:.4f}")
print(f"Fold별 AUC: {[round(s, 4) for s in fold_scores]}")


📥 데이터를 불러오고 K-Fold를 위해 병합합니다...


[I 2026-04-09 10:55:34,693] A new study created in memory with name: no-name-17b98db8-cdda-4a1e-825c-46b5f7787184


✅ 병합 완료! (전체 학습 데이터 X: (200000, 44), y: (200000,))

🔍 Optuna 탐색을 시작합니다 (탐색 속도를 위해 임시 8:2 분할)...


[I 2026-04-09 10:55:44,028] Trial 0 finished with value: 0.5969420475255263 and parameters: {'iterations': 491, 'depth': 6, 'learning_rate': 0.023640934364281555, 'l2_leaf_reg': 1.6035256491292313}. Best is trial 0 with value: 0.5969420475255263.
[I 2026-04-09 10:56:15,008] Trial 1 finished with value: 0.5952177878319906 and parameters: {'iterations': 870, 'depth': 9, 'learning_rate': 0.05097748670839996, 'l2_leaf_reg': 6.208972076404386}. Best is trial 0 with value: 0.5969420475255263.
[I 2026-04-09 10:57:31,001] Trial 2 finished with value: 0.5954676376043072 and parameters: {'iterations': 711, 'depth': 10, 'learning_rate': 0.013750017647355839, 'l2_leaf_reg': 2.196062405051314}. Best is trial 0 with value: 0.5969420475255263.
[I 2026-04-09 10:59:09,228] Trial 3 finished with value: 0.5955967482925135 and parameters: {'iterations': 917, 'depth': 10, 'learning_rate': 0.017932537041771758, 'l2_leaf_reg': 6.0219328390726}. Best is trial 0 with value: 0.5969420475255263.
[I 2026-04-09 10

🎉 탐색 완료! 최고 Validation AUC: 0.5972

🚀 CatBoost K-Fold 학습을 시작합니다...
   - Fold 1 완료 (ROC-AUC: 0.5971)
   - Fold 2 완료 (ROC-AUC: 0.5958)
   - Fold 3 완료 (ROC-AUC: 0.5979)
   - Fold 4 완료 (ROC-AUC: 0.5879)
   - Fold 5 완료 (ROC-AUC: 0.5934)

--- 📊 최종 Validation(OOF) 결과 ---
정확도 (Accuracy): 0.5698
F1 점수 (F1 Score): 0.4857
AUC (ROC-AUC):  0.5944
Fold별 AUC: [0.5971, 0.5958, 0.5979, 0.5879, 0.5934]


## RandomForest

In [10]:
import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score
from sklearn.ensemble import RandomForestClassifier

# ==========================================
# 1. 데이터 불러오기 (트리 기반 모델용 unscaled)
# ==========================================
# XGBoost는 트리 기반 모델이므로 스케일링이 안 된 데이터를 부릅니다.
print("📥 데이터를 불러오고 K-Fold를 위해 병합합니다...")

# 기존에 쪼개져 있던 Train과 Val 데이터를 불러옵니다.
X_train_part = pd.read_csv('X_tr_unscaled.csv')
X_val_part = pd.read_csv('X_val_unscaled.csv')

y_train_part = pd.read_csv('y_tr.csv')['returned']
y_val_part = pd.read_csv('y_val.csv')['returned']

# K-Fold가 전체 데이터에서 자유롭게 쪼갤 수 있도록 하나로 합칩니다.
X = pd.concat([X_train_part, X_val_part], axis=0).reset_index(drop=True)
y = pd.concat([y_train_part, y_val_part], axis=0).reset_index(drop=True)

X_test = pd.read_csv('X_test_unscaled.csv')
test_id_df = pd.read_csv('test_order_id.csv')

print(f"✅ 병합 완료! (전체 학습 데이터 X: {X.shape}, y: {y.shape})")

# ==========================================
# 🌟 1.5 K-Fold 시작 전 Optuna 하이퍼파라미터 탐색 추가
# ==========================================
import optuna
from sklearn.model_selection import train_test_split

print("\n🔍 Optuna 탐색을 시작합니다 (탐색 속도를 위해 임시 8:2 분할)...")
# 전체 X, y를 Optuna가 빨리 학습해볼 수 있게 임시로 분할합니다.
X_opt_tr, X_opt_val, y_opt_tr, y_opt_val = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
scale_weight_opt = (y_opt_tr == 0).sum() / (y_opt_tr == 1).sum()

def objective(trial):
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 100, 500),
        'max_depth': trial.suggest_int('max_depth', 10, 30),
        'min_samples_split': trial.suggest_int('min_samples_split', 2, 20),
        'class_weight': 'balanced',
        'random_state': 42,
        'n_jobs': -1
    }
    
    model_opt = RandomForestClassifier(**params)
    
    # 🚨 주의: Random Forest는 eval_set이 없으므로 깔끔하게 X와 y만 넣습니다!
    model_opt.fit(X_opt_tr, y_opt_tr)
    
    return roc_auc_score(y_opt_val, model_opt.predict_proba(X_opt_val)[:, 1])

# 30회 탐색 실행
study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=30)
print(f"🎉 탐색 완료! 최고 Validation AUC: {study.best_value:.4f}")

# ==========================================
# 2. K-Fold 설정 및 XGBoost 모델 학습
# ==========================================
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

oof_pred = np.zeros(len(X))          # 5번 검증한 결과를 모아둘 빈 배열
test_pred = np.zeros(len(X_test))    # 5번 예측한 테스트 결과를 누적할 빈 배열
fold_scores = []

print("\n🚀 XGBoost K-Fold 학습을 시작합니다...")

for fold, (train_idx, val_idx) in enumerate(skf.split(X, y), 1):
    # Fold별 데이터 분할
    X_tr, X_va = X.iloc[train_idx], X.iloc[val_idx]
    y_tr, y_va = y.iloc[train_idx], y.iloc[val_idx]
    
    # 기존에 세팅하셨던 모델 파라미터 그대로 적용
    # Fold별 클래스 불균형 비율 계산
    scale_pos_weight_fold = (y_tr == 0).sum() / (y_tr == 1).sum()

# 🌟 Optuna 결과 덮어쓰기
    model = RandomForestClassifier(
        **study.best_params,
        class_weight='balanced',    # RF용 불균형 처리
        random_state=42 + fold,     
        n_jobs=-1,
    )
    
    # 🚨 주의: 여기서도 eval_set을 완전히 제거해야 에러가 나지 않습니다!
    model.fit(X_tr, y_tr)
    
    # 검증 데이터 예측 및 OOF 저장
    val_proba = model.predict_proba(X_va)[:, 1]
    oof_pred[val_idx] = val_proba
    
    # 테스트 데이터 예측 (5로 나누어 누적 합산 -> 최종적으로 평균이 됨)
    test_pred += model.predict_proba(X_test)[:, 1] / skf.n_splits
    
    # Fold별 점수 기록
    fold_auc = roc_auc_score(y_va, val_proba)
    fold_scores.append(fold_auc)
    print(f"   - Fold {fold} 완료 (ROC-AUC: {fold_auc:.4f})")

# ==========================================
# 3. 전체 OOF 성능 평가
# ==========================================
overall_auc = roc_auc_score(y, oof_pred)
# 기본 임계값 0.5를 기준으로 F1 및 정확도 계산
oof_label = (oof_pred >= 0.5).astype(int)
overall_f1 = f1_score(y, oof_label)
overall_acc = accuracy_score(y, oof_label)

print("\n--- 📊 최종 Validation(OOF) 결과 ---")
print(f"정확도 (Accuracy): {overall_acc:.4f}")
print(f"F1 점수 (F1 Score): {overall_f1:.4f}")
print(f"AUC (ROC-AUC):  {overall_auc:.4f}")
print(f"Fold별 AUC: {[round(s, 4) for s in fold_scores]}")


📥 데이터를 불러오고 K-Fold를 위해 병합합니다...


[I 2026-04-09 11:13:45,651] A new study created in memory with name: no-name-db563d36-0c3c-4b58-b276-78b15604e06b


✅ 병합 완료! (전체 학습 데이터 X: (200000, 44), y: (200000,))

🔍 Optuna 탐색을 시작합니다 (탐색 속도를 위해 임시 8:2 분할)...


[I 2026-04-09 11:14:09,750] Trial 0 finished with value: 0.5754201021323668 and parameters: {'n_estimators': 213, 'max_depth': 28, 'min_samples_split': 3}. Best is trial 0 with value: 0.5754201021323668.
[I 2026-04-09 11:14:39,206] Trial 1 finished with value: 0.5825625567364163 and parameters: {'n_estimators': 274, 'max_depth': 22, 'min_samples_split': 20}. Best is trial 1 with value: 0.5825625567364163.
[I 2026-04-09 11:15:15,688] Trial 2 finished with value: 0.5785104522134 and parameters: {'n_estimators': 351, 'max_depth': 27, 'min_samples_split': 8}. Best is trial 1 with value: 0.5825625567364163.
[I 2026-04-09 11:15:32,862] Trial 3 finished with value: 0.5911114959709625 and parameters: {'n_estimators': 241, 'max_depth': 12, 'min_samples_split': 13}. Best is trial 3 with value: 0.5911114959709625.
[I 2026-04-09 11:16:01,452] Trial 4 finished with value: 0.5920284497986885 and parameters: {'n_estimators': 428, 'max_depth': 10, 'min_samples_split': 5}. Best is trial 4 with value: 0

🎉 탐색 완료! 최고 Validation AUC: 0.5921

🚀 XGBoost K-Fold 학습을 시작합니다...
   - Fold 1 완료 (ROC-AUC: 0.5919)
   - Fold 2 완료 (ROC-AUC: 0.5910)
   - Fold 3 완료 (ROC-AUC: 0.5950)
   - Fold 4 완료 (ROC-AUC: 0.5841)
   - Fold 5 완료 (ROC-AUC: 0.5859)

--- 📊 최종 Validation(OOF) 결과 ---
정확도 (Accuracy): 0.5634
F1 점수 (F1 Score): 0.5434
AUC (ROC-AUC):  0.5896
Fold별 AUC: [0.5919, 0.591, 0.595, 0.5841, 0.5859]


## Logistic

In [12]:
import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score
from sklearn.linear_model import LogisticRegression

# ==========================================
# 1. 데이터 불러오기 (트리 기반 모델용 unscaled)
# ==========================================
# XGBoost는 트리 기반 모델이므로 스케일링이 안 된 데이터를 부릅니다.
print("📥 데이터를 불러오고 K-Fold를 위해 병합합니다...")

# 기존에 쪼개져 있던 Train과 Val 데이터를 불러옵니다.
# 🚨 반드시 스케일링된 데이터를 불러와야 합니다!
X_train_part = pd.read_csv('X_tr_scaled.csv')
X_val_part = pd.read_csv('X_val_scaled.csv')

y_train_part = pd.read_csv('y_tr.csv')['returned']
y_val_part = pd.read_csv('y_val.csv')['returned']

# K-Fold가 전체 데이터에서 자유롭게 쪼갤 수 있도록 하나로 합칩니다.
X = pd.concat([X_train_part, X_val_part], axis=0).reset_index(drop=True)
y = pd.concat([y_train_part, y_val_part], axis=0).reset_index(drop=True)

X_test = pd.read_csv('X_test_scaled.csv')
test_id_df = pd.read_csv('test_order_id.csv')

print(f"✅ 병합 완료! (전체 학습 데이터 X: {X.shape}, y: {y.shape})")

# ==========================================
# 🌟 1.5 K-Fold 시작 전 Optuna 하이퍼파라미터 탐색 추가
# ==========================================
import optuna
from sklearn.model_selection import train_test_split

print("\n🔍 Optuna 탐색을 시작합니다 (탐색 속도를 위해 임시 8:2 분할)...")
# 전체 X, y를 Optuna가 빨리 학습해볼 수 있게 임시로 분할합니다.
X_opt_tr, X_opt_val, y_opt_tr, y_opt_val = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
scale_weight_opt = (y_opt_tr == 0).sum() / (y_opt_tr == 1).sum()

def objective(trial):
    params = {
        # C값이 작을수록 규제가 강해짐 (과적합 방지)
        'C': trial.suggest_float('C', 0.001, 10.0, log=True), 
        'solver': trial.suggest_categorical('solver', ['lbfgs', 'liblinear']),
        'class_weight': 'balanced',
        'max_iter': 1000, # 에러 방지를 위해 반복 횟수를 넉넉히 줍니다
        'random_state': 42
    }
    
    model_opt = LogisticRegression(**params)
    
    # 🚨 로지스틱은 eval_set이 없습니다.
    model_opt.fit(X_opt_tr, y_opt_tr)
    
    return roc_auc_score(y_opt_val, model_opt.predict_proba(X_opt_val)[:, 1])

# 30회 탐색 실행
study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=30)
print(f"🎉 탐색 완료! 최고 Validation AUC: {study.best_value:.4f}")

# ==========================================
# 2. K-Fold 설정 및 XGBoost 모델 학습
# ==========================================
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

oof_pred = np.zeros(len(X))          # 5번 검증한 결과를 모아둘 빈 배열
test_pred = np.zeros(len(X_test))    # 5번 예측한 테스트 결과를 누적할 빈 배열
fold_scores = []

print("\n🚀 Logistic K-Fold 학습을 시작합니다...")

for fold, (train_idx, val_idx) in enumerate(skf.split(X, y), 1):
    # Fold별 데이터 분할
    X_tr, X_va = X.iloc[train_idx], X.iloc[val_idx]
    y_tr, y_va = y.iloc[train_idx], y.iloc[val_idx]
    
    # 기존에 세팅하셨던 모델 파라미터 그대로 적용
    # Fold별 클래스 불균형 비율 계산
    scale_pos_weight_fold = (y_tr == 0).sum() / (y_tr == 1).sum()

# 🌟 Optuna 결과 덮어쓰기
    model = LogisticRegression(
        **study.best_params,
        class_weight='balanced',
        random_state=42 + fold,
        max_iter=1000
    )
    
    # 🚨 eval_set 제거
    model.fit(X_tr, y_tr)
    
    # 검증 데이터 예측 및 OOF 저장
    val_proba = model.predict_proba(X_va)[:, 1]
    oof_pred[val_idx] = val_proba
    
    # 테스트 데이터 예측 (5로 나누어 누적 합산 -> 최종적으로 평균이 됨)
    test_pred += model.predict_proba(X_test)[:, 1] / skf.n_splits
    
    # Fold별 점수 기록
    fold_auc = roc_auc_score(y_va, val_proba)
    fold_scores.append(fold_auc)
    print(f"   - Fold {fold} 완료 (ROC-AUC: {fold_auc:.4f})")

# ==========================================
# 3. 전체 OOF 성능 평가
# ==========================================
overall_auc = roc_auc_score(y, oof_pred)
# 기본 임계값 0.5를 기준으로 F1 및 정확도 계산
oof_label = (oof_pred >= 0.5).astype(int)
overall_f1 = f1_score(y, oof_label)
overall_acc = accuracy_score(y, oof_label)

print("\n--- 📊 최종 Validation(OOF) 결과 ---")
print(f"정확도 (Accuracy): {overall_acc:.4f}")
print(f"F1 점수 (F1 Score): {overall_f1:.4f}")
print(f"AUC (ROC-AUC):  {overall_auc:.4f}")
print(f"Fold별 AUC: {[round(s, 4) for s in fold_scores]}")


📥 데이터를 불러오고 K-Fold를 위해 병합합니다...


[I 2026-04-09 11:42:00,503] A new study created in memory with name: no-name-dcc544ab-6f3c-4e8a-8138-c8f2cd9eef09


✅ 병합 완료! (전체 학습 데이터 X: (200000, 44), y: (200000,))

🔍 Optuna 탐색을 시작합니다 (탐색 속도를 위해 임시 8:2 분할)...


[I 2026-04-09 11:42:02,250] Trial 0 finished with value: 0.589183921595426 and parameters: {'C': 0.19044542117849222, 'solver': 'liblinear'}. Best is trial 0 with value: 0.589183921595426.
[I 2026-04-09 11:42:02,629] Trial 1 finished with value: 0.5891953310390927 and parameters: {'C': 0.003958582799166427, 'solver': 'lbfgs'}. Best is trial 1 with value: 0.5891953310390927.
[I 2026-04-09 11:42:02,969] Trial 2 finished with value: 0.5891875710132597 and parameters: {'C': 0.030379933714944764, 'solver': 'lbfgs'}. Best is trial 1 with value: 0.5891953310390927.
[I 2026-04-09 11:42:03,317] Trial 3 finished with value: 0.5891956919705268 and parameters: {'C': 0.0037672208345125103, 'solver': 'lbfgs'}. Best is trial 3 with value: 0.5891956919705268.
[I 2026-04-09 11:42:04,937] Trial 4 finished with value: 0.5891959100332682 and parameters: {'C': 0.022160058214975124, 'solver': 'liblinear'}. Best is trial 4 with value: 0.5891959100332682.
[I 2026-04-09 11:42:05,319] Trial 5 finished with valu

🎉 탐색 완료! 최고 Validation AUC: 0.5892

🚀 Logistic K-Fold 학습을 시작합니다...
   - Fold 1 완료 (ROC-AUC: 0.5910)
   - Fold 2 완료 (ROC-AUC: 0.5884)
   - Fold 3 완료 (ROC-AUC: 0.5927)
   - Fold 4 완료 (ROC-AUC: 0.5823)
   - Fold 5 완료 (ROC-AUC: 0.5858)

--- 📊 최종 Validation(OOF) 결과 ---
정확도 (Accuracy): 0.5630
F1 점수 (F1 Score): 0.5471
AUC (ROC-AUC):  0.5880
Fold별 AUC: [0.591, 0.5884, 0.5927, 0.5823, 0.5858]


# KNN

In [16]:
import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score
from sklearn.neighbors import KNeighborsClassifier

# ==========================================
# 1. 데이터 불러오기 (트리 기반 모델용 unscaled)
# ==========================================
# XGBoost는 트리 기반 모델이므로 스케일링이 안 된 데이터를 부릅니다.
print("📥 데이터를 불러오고 K-Fold를 위해 병합합니다...")

# 기존에 쪼개져 있던 Train과 Val 데이터를 불러옵니다.
X_train_part = pd.read_csv('X_tr_scaled.csv')
X_val_part = pd.read_csv('X_val_scaled.csv')

y_train_part = pd.read_csv('y_tr.csv')['returned']
y_val_part = pd.read_csv('y_val.csv')['returned']

# K-Fold가 전체 데이터에서 자유롭게 쪼갤 수 있도록 하나로 합칩니다.
X = pd.concat([X_train_part, X_val_part], axis=0).reset_index(drop=True)
y = pd.concat([y_train_part, y_val_part], axis=0).reset_index(drop=True)

X_test = pd.read_csv('X_test_scaled.csv')
test_id_df = pd.read_csv('test_order_id.csv')

print(f"✅ 병합 완료! (전체 학습 데이터 X: {X.shape}, y: {y.shape})")

# ==========================================
# 🌟 1.5 K-Fold 시작 전 Optuna 하이퍼파라미터 탐색 추가
# ==========================================
import optuna
from sklearn.model_selection import train_test_split

print("\n🔍 Optuna 탐색을 시작합니다 (탐색 속도를 위해 임시 8:2 분할)...")
# 전체 X, y를 Optuna가 빨리 학습해볼 수 있게 임시로 분할합니다.
X_opt_tr, X_opt_val, y_opt_tr, y_opt_val = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
scale_weight_opt = (y_opt_tr == 0).sum() / (y_opt_tr == 1).sum()

def objective(trial):
    params = {
        'n_neighbors': trial.suggest_int('n_neighbors', 5, 50),
        'weights': trial.suggest_categorical('weights', ['uniform', 'distance']),
        'p': trial.suggest_categorical('p', [1, 2]) # 1: 맨해튼 거리, 2: 유클리디안 거리
    }
    model_opt = KNeighborsClassifier(**params)
    model_opt.fit(X_opt_tr, y_opt_tr)
    return roc_auc_score(y_opt_val, model_opt.predict_proba(X_opt_val)[:, 1])

# 30회 탐색 실행
study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=30)
print(f"🎉 탐색 완료! 최고 Validation AUC: {study.best_value:.4f}")

# ==========================================
# 2. K-Fold 설정 및 XGBoost 모델 학습
# ==========================================
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

oof_pred = np.zeros(len(X))          # 5번 검증한 결과를 모아둘 빈 배열
test_pred = np.zeros(len(X_test))    # 5번 예측한 테스트 결과를 누적할 빈 배열
fold_scores = []

print("\n🚀 XGBoost K-Fold 학습을 시작합니다...")

for fold, (train_idx, val_idx) in enumerate(skf.split(X, y), 1):
    # Fold별 데이터 분할
    X_tr, X_va = X.iloc[train_idx], X.iloc[val_idx]
    y_tr, y_va = y.iloc[train_idx], y.iloc[val_idx]
    
    # 기존에 세팅하셨던 모델 파라미터 그대로 적용
    # Fold별 클래스 불균형 비율 계산
    scale_pos_weight_fold = (y_tr == 0).sum() / (y_tr == 1).sum()

    # 🌟 Optuna 결과 덮어쓰기
    model = KNeighborsClassifier(
        **study.best_params,
        n_jobs=-1
        # 🚨 주의: KNN은 random_state나 class_weight 파라미터가 없으므로 지워야 에러가 안 납니다!
    )
    
    # 🚨 주의: eval_set 없음
    model.fit(X_tr, y_tr)
    
    
    # 검증 데이터 예측 및 OOF 저장
    val_proba = model.predict_proba(X_va)[:, 1]
    oof_pred[val_idx] = val_proba
    
    # 테스트 데이터 예측 (5로 나누어 누적 합산 -> 최종적으로 평균이 됨)
    test_pred += model.predict_proba(X_test)[:, 1] / skf.n_splits
    
    # Fold별 점수 기록
    fold_auc = roc_auc_score(y_va, val_proba)
    fold_scores.append(fold_auc)
    print(f"   - Fold {fold} 완료 (ROC-AUC: {fold_auc:.4f})")

# ==========================================
# 3. 전체 OOF 성능 평가
# ==========================================
overall_auc = roc_auc_score(y, oof_pred)
# 기본 임계값 0.5를 기준으로 F1 및 정확도 계산
oof_label = (oof_pred >= 0.5).astype(int)
overall_f1 = f1_score(y, oof_label)
overall_acc = accuracy_score(y, oof_label)

print("\n--- 📊 최종 Validation(OOF) 결과 ---")
print(f"정확도 (Accuracy): {overall_acc:.4f}")
print(f"F1 점수 (F1 Score): {overall_f1:.4f}")
print(f"AUC (ROC-AUC):  {overall_auc:.4f}")
print(f"Fold별 AUC: {[round(s, 4) for s in fold_scores]}")


📥 데이터를 불러오고 K-Fold를 위해 병합합니다...


[I 2026-04-09 12:11:01,117] A new study created in memory with name: no-name-d10ace2e-7ed9-4665-80c6-a8e890ed2992


✅ 병합 완료! (전체 학습 데이터 X: (200000, 44), y: (200000,))

🔍 Optuna 탐색을 시작합니다 (탐색 속도를 위해 임시 8:2 분할)...


[I 2026-04-09 12:11:12,412] Trial 0 finished with value: 0.549226596123019 and parameters: {'n_neighbors': 41, 'weights': 'uniform', 'p': 2}. Best is trial 0 with value: 0.549226596123019.
[I 2026-04-09 12:11:51,979] Trial 1 finished with value: 0.5504857192164387 and parameters: {'n_neighbors': 30, 'weights': 'uniform', 'p': 1}. Best is trial 1 with value: 0.5504857192164387.
[I 2026-04-09 12:11:59,701] Trial 2 finished with value: 0.5494158344790901 and parameters: {'n_neighbors': 40, 'weights': 'distance', 'p': 2}. Best is trial 1 with value: 0.5504857192164387.
[I 2026-04-09 12:12:38,349] Trial 3 finished with value: 0.5527807242983533 and parameters: {'n_neighbors': 36, 'weights': 'uniform', 'p': 1}. Best is trial 3 with value: 0.5527807242983533.
[I 2026-04-09 12:13:17,911] Trial 4 finished with value: 0.550054980133933 and parameters: {'n_neighbors': 27, 'weights': 'uniform', 'p': 1}. Best is trial 3 with value: 0.5527807242983533.
[I 2026-04-09 12:13:25,557] Trial 5 finished wi

🎉 탐색 완료! 최고 Validation AUC: 0.5584

🚀 XGBoost K-Fold 학습을 시작합니다...
   - Fold 1 완료 (ROC-AUC: 0.5544)
   - Fold 2 완료 (ROC-AUC: 0.5531)
   - Fold 3 완료 (ROC-AUC: 0.5556)
   - Fold 4 완료 (ROC-AUC: 0.5506)
   - Fold 5 완료 (ROC-AUC: 0.5504)

--- 📊 최종 Validation(OOF) 결과 ---
정확도 (Accuracy): 0.5415
F1 점수 (F1 Score): 0.4507
AUC (ROC-AUC):  0.5528
Fold별 AUC: [0.5544, 0.5531, 0.5556, 0.5506, 0.5504]


# ExTrees

In [14]:
import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score
from sklearn.ensemble import ExtraTreesClassifier

# ==========================================
# 1. 데이터 불러오기 (트리 기반 모델용 unscaled)
# ==========================================
# XGBoost는 트리 기반 모델이므로 스케일링이 안 된 데이터를 부릅니다.
print("📥 데이터를 불러오고 K-Fold를 위해 병합합니다...")

# 기존에 쪼개져 있던 Train과 Val 데이터를 불러옵니다.
X_train_part = pd.read_csv('X_tr_unscaled.csv')
X_val_part = pd.read_csv('X_val_unscaled.csv')

y_train_part = pd.read_csv('y_tr.csv')['returned']
y_val_part = pd.read_csv('y_val.csv')['returned']

# K-Fold가 전체 데이터에서 자유롭게 쪼갤 수 있도록 하나로 합칩니다.
X = pd.concat([X_train_part, X_val_part], axis=0).reset_index(drop=True)
y = pd.concat([y_train_part, y_val_part], axis=0).reset_index(drop=True)

X_test = pd.read_csv('X_test_unscaled.csv')
test_id_df = pd.read_csv('test_order_id.csv')

print(f"✅ 병합 완료! (전체 학습 데이터 X: {X.shape}, y: {y.shape})")

# ==========================================
# 🌟 1.5 K-Fold 시작 전 Optuna 하이퍼파라미터 탐색 추가
# ==========================================
import optuna
from sklearn.model_selection import train_test_split

print("\n🔍 Optuna 탐색을 시작합니다 (탐색 속도를 위해 임시 8:2 분할)...")
# 전체 X, y를 Optuna가 빨리 학습해볼 수 있게 임시로 분할합니다.
X_opt_tr, X_opt_val, y_opt_tr, y_opt_val = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
scale_weight_opt = (y_opt_tr == 0).sum() / (y_opt_tr == 1).sum()

def objective(trial):
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 100, 500),
        'max_depth': trial.suggest_int('max_depth', 10, 30),
        'min_samples_split': trial.suggest_int('min_samples_split', 2, 20),
        'class_weight': 'balanced',
        'random_state': 42,
        'n_jobs': -1
    }
    
    model_opt = ExtraTreesClassifier(**params)
    
    # 🚨 주의: eval_set이 없으므로 통째로 뺍니다!
    model_opt.fit(X_opt_tr, y_opt_tr)
    
    return roc_auc_score(y_opt_val, model_opt.predict_proba(X_opt_val)[:, 1])

# 30회 탐색 실행
study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=30)
print(f"🎉 탐색 완료! 최고 Validation AUC: {study.best_value:.4f}")

# ==========================================
# 2. K-Fold 설정 및 XGBoost 모델 학습
# ==========================================
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

oof_pred = np.zeros(len(X))          # 5번 검증한 결과를 모아둘 빈 배열
test_pred = np.zeros(len(X_test))    # 5번 예측한 테스트 결과를 누적할 빈 배열
fold_scores = []

print("\n🚀 XGBoost K-Fold 학습을 시작합니다...")

for fold, (train_idx, val_idx) in enumerate(skf.split(X, y), 1):
    # Fold별 데이터 분할
    X_tr, X_va = X.iloc[train_idx], X.iloc[val_idx]
    y_tr, y_va = y.iloc[train_idx], y.iloc[val_idx]
    
    # 기존에 세팅하셨던 모델 파라미터 그대로 적용
    # Fold별 클래스 불균형 비율 계산
    scale_pos_weight_fold = (y_tr == 0).sum() / (y_tr == 1).sum()

# 🌟 Optuna 결과 덮어쓰기
    model = ExtraTreesClassifier(
        **study.best_params,
        class_weight='balanced',
        random_state=42 + fold,
        n_jobs=-1
    )
    
    # 🚨 주의: 여기서도 eval_set을 뺍니다!
    model.fit(X_tr, y_tr)
    
    # 검증 데이터 예측 및 OOF 저장
    val_proba = model.predict_proba(X_va)[:, 1]
    oof_pred[val_idx] = val_proba
    
    # 테스트 데이터 예측 (5로 나누어 누적 합산 -> 최종적으로 평균이 됨)
    test_pred += model.predict_proba(X_test)[:, 1] / skf.n_splits
    
    # Fold별 점수 기록
    fold_auc = roc_auc_score(y_va, val_proba)
    fold_scores.append(fold_auc)
    print(f"   - Fold {fold} 완료 (ROC-AUC: {fold_auc:.4f})")

# ==========================================
# 3. 전체 OOF 성능 평가
# ==========================================
overall_auc = roc_auc_score(y, oof_pred)
# 기본 임계값 0.5를 기준으로 F1 및 정확도 계산
oof_label = (oof_pred >= 0.5).astype(int)
overall_f1 = f1_score(y, oof_label)
overall_acc = accuracy_score(y, oof_label)

print("\n--- 📊 최종 Validation(OOF) 결과 ---")
print(f"정확도 (Accuracy): {overall_acc:.4f}")
print(f"F1 점수 (F1 Score): {overall_f1:.4f}")
print(f"AUC (ROC-AUC):  {overall_auc:.4f}")
print(f"Fold별 AUC: {[round(s, 4) for s in fold_scores]}")


📥 데이터를 불러오고 K-Fold를 위해 병합합니다...
✅ 병합 완료! (전체 학습 데이터 X: (200000, 44), y: (200000,))

🔍 Optuna 탐색을 시작합니다 (탐색 속도를 위해 임시 8:2 분할)...


[I 2026-04-09 11:44:19,989] A new study created in memory with name: no-name-823b133a-1604-485e-b428-0b67701c7fac
[I 2026-04-09 11:44:39,025] Trial 0 finished with value: 0.5896912332842628 and parameters: {'n_estimators': 398, 'max_depth': 11, 'min_samples_split': 5}. Best is trial 0 with value: 0.5896912332842628.
[I 2026-04-09 11:45:02,729] Trial 1 finished with value: 0.583005572500618 and parameters: {'n_estimators': 359, 'max_depth': 20, 'min_samples_split': 11}. Best is trial 0 with value: 0.5896912332842628.
[I 2026-04-09 11:45:27,427] Trial 2 finished with value: 0.5807686322631636 and parameters: {'n_estimators': 369, 'max_depth': 25, 'min_samples_split': 19}. Best is trial 0 with value: 0.5896912332842628.
[I 2026-04-09 11:45:51,100] Trial 3 finished with value: 0.5854919940595498 and parameters: {'n_estimators': 376, 'max_depth': 17, 'min_samples_split': 4}. Best is trial 0 with value: 0.5896912332842628.
[I 2026-04-09 11:46:07,239] Trial 4 finished with value: 0.5893300537

🎉 탐색 완료! 최고 Validation AUC: 0.5899

🚀 XGBoost K-Fold 학습을 시작합니다...
   - Fold 1 완료 (ROC-AUC: 0.5925)
   - Fold 2 완료 (ROC-AUC: 0.5888)
   - Fold 3 완료 (ROC-AUC: 0.5924)
   - Fold 4 완료 (ROC-AUC: 0.5836)
   - Fold 5 완료 (ROC-AUC: 0.5858)

--- 📊 최종 Validation(OOF) 결과 ---
정확도 (Accuracy): 0.5625
F1 점수 (F1 Score): 0.5455
AUC (ROC-AUC):  0.5886
Fold별 AUC: [0.5925, 0.5888, 0.5924, 0.5836, 0.5858]


## CSV 추출 코드 (대시보드 투입용)

In [4]:
import pandas as pd

# (이전 단계: 모델 학습 및 val_pred, val_proba, test_proba가 계산된 상태라고 가정)
# 만약 test_pred 계산이 안 되어 있다면 아래 한 줄을 먼저 추가합니다.
test_pred = model.predict(X_test)

print("🚀 대시보드용 CSV 파일 추출을 시작합니다...")

# ==========================================
# 1. Validation 예측 결과 (val_predictions.csv)
# ==========================================
# 팀원 요청: pred_prob, pred_label, returned (순서 섞임 없이 그대로)
# 주의: val_order_id.csv가 없으므로 order_id는 제외됩니다.
val_df = pd.DataFrame({
    'pred_prob': val_proba,
    'pred_label': val_pred,
    # y_val이 Series 형태일 수 있으므로 .values를 써서 순서대로 값만 깔끔하게 매칭합니다.
    'returned': y_val.values 
})

val_df.to_csv('val_predictions.csv', index=False)
print("✅ [1/3] 'val_predictions.csv' 생성 완료!")

# ==========================================
# 2. Test 예측 결과 (test_predictions.csv)
# ==========================================
# 팀원 요청: order_id, pred_prob, pred_label
test_id_df = pd.read_csv('test_order_id.csv')

test_df = pd.DataFrame({
    'order_id': test_id_df['order_id'],
    'pred_prob': test_proba,
    'pred_label': test_pred
})

test_df.to_csv('test_predictions.csv', index=False)
print("✅ [2/3] 'test_predictions.csv' 생성 완료!")

# ==========================================
# 3. 변수 중요도 (feature_importance.csv)
# ==========================================
# 팀원 요청: feature, importance
importance_df = pd.DataFrame({
    'feature': X_train.columns,
    'importance': model.feature_importances_
})

# 중요도가 높은 순서대로 정렬하여 대시보드에서 바로 쓰기 좋게 만듭니다.
importance_df = importance_df.sort_values(by='importance', ascending=False)

importance_df.to_csv('feature_importance.csv', index=False)
print("✅ [3/3] 'feature_importance.csv' 생성 완료!")

print("\n🎉 모든 데이터 추출이 완료되었습니다! 대시보드 담당자에게 전달해 주세요.")


🚀 대시보드용 CSV 파일 추출을 시작합니다...


NameError: name 'val_pred' is not defined

In [17]:
import pandas as pd

print("🚀 대시보드용 CSV 파일 추출을 시작합니다...")

# ==========================================
# 1. Validation 예측 결과 (val_predictions.csv)
# ==========================================
val_df = pd.DataFrame({
    'pred_prob': oof_pred,
    'pred_label': oof_label,
    'returned': y.values 
})

val_df.to_csv('val_predictions.csv', index=False)
print("✅ [1/3] 'val_predictions.csv' 생성 완료!")

# ==========================================
# 2. Test 예측 결과 (test_predictions.csv)
# ==========================================
test_label = (test_pred >= 0.5).astype(int)

test_df = pd.DataFrame({
    'order_id': test_id_df['order_id'],
    'pred_prob': test_pred,
    'pred_label': test_label
})

test_df.to_csv('test_predictions.csv', index=False)
print("✅ [2/3] 'test_predictions.csv' 생성 완료!")

# ==========================================
# 3. 변수 중요도 (feature_importance.csv)
# ==========================================
# (주의: PyTorch나 로지스틱을 돌리셨을 때는 이 부분이 해당 모델 전용 코드로 살짝 바뀝니다)
importance_df = pd.DataFrame({
    'feature': X.columns,
    'importance': model.feature_importances_
})

importance_df = importance_df.sort_values(by='importance', ascending=False)
importance_df.to_csv('feature_importance.csv', index=False)
print("✅ [3/3] 'feature_importance.csv' 생성 완료!")

# ---------------------------------------------------------
# 💡 [핵심] 대시보드(app3.py) 에러 방지용 데이터 덮어쓰기
# K-Fold 결과와 길이가 맞지 않아 app3.py가 튕기는 것을 막아줍니다.
# (출력 메시지에는 띄우지 않고 조용히 파일만 교체합니다)
X.to_csv('X_val_unscaled.csv', index=False)
pd.DataFrame({'returned': y}).to_csv('y_val.csv', index=False)

# 캐글 제출용 파일도 하나 조용히 뽑아둡니다.
test_df[['order_id', 'pred_prob']].rename(columns={'pred_prob': 'returned'}).to_csv('xgb_submission.csv', index=False)
# ---------------------------------------------------------

print("\n🎉 모든 데이터 추출이 완료되었습니다! 터미널에서 `streamlit run app3.py`를 실행해 보세요.")

🚀 대시보드용 CSV 파일 추출을 시작합니다...
✅ [1/3] 'val_predictions.csv' 생성 완료!
✅ [2/3] 'test_predictions.csv' 생성 완료!


AttributeError: 'KNeighborsClassifier' object has no attribute 'feature_importances_'